In [1]:
import pandas as pd
import plotly.express as px 
import plotly.graph_objects as go
import numpy as np
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots

In [2]:
DELAY_DATA_URL = 'https://full-stack-assets.s3.eu-west-3.amazonaws.com/Deployment/get_around_delay_analysis.xlsx'

In [3]:
data = pd.read_excel(DELAY_DATA_URL)
#data = data.rename(columns={'checkin_type':'Type de checking', 'state': 'Satus'})
data.head()


,rental_id,car_id,checkin_type,state,delay_at_checkout_in_minutes,previous_ended_rental_id,time_delta_with_previous_rental_in_minutes
0,505000,363965,mobile,canceled,NaN,NaN,NaN
1,507750,269550,mobile,ended,-81.0,NaN,NaN
2,508131,359049,connect,ended,70.0,NaN,NaN
3,508865,299063,connect,canceled,NaN,NaN,NaN
4,511440,313932,mobile,ended,NaN,NaN,NaN


In [4]:
palette = {
    "mobile": px.colors.qualitative.Plotly[0],
    "connect": px.colors.qualitative.Plotly[1],
    "mobile + ended": px.colors.qualitative.Plotly[2],
    "mobile + canceled": px.colors.qualitative.Plotly[3],
    "connect + ended": px.colors.qualitative.Plotly[4],
    "connect + canceled": px.colors.qualitative.Plotly[5],
    "Total": px.colors.qualitative.Plotly[8]
}

In [5]:
fig = px.histogram(data, x="checkin_type", color="state", title='Répartition des locations par type et par status', text_auto='.2f',
                   histnorm='percent',
                   barmode="group",
                   #barnorm='percent',
                   labels={
                     "percent": "Pourcentage",
                     'checkin_type':'Type de checking',
                     'state': 'Satus'
                 },             

                 width = 500,
                 height = 500)
fig.show()


In [6]:
#répartition des locations
#histnorm='percent'
histnorm=''
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.25, 0.75],  # smaller box plot on top
    vertical_spacing=0.02
)
for checkin_type, subset in data.groupby("checkin_type"):
    for state, subset2 in subset.groupby('state'):
        counts = subset2.groupby("car_id").size()
        legend_name = f'{checkin_type} + {state}'
        base_color = palette.get(legend_name, "gray") 
        fig.add_trace(
            go.Histogram(
                x=counts,
                name=str(legend_name).capitalize(),
                histnorm=histnorm,
                marker_color=base_color,
                showlegend=False,
                nbinsx=25,
                opacity=0.6 
            ),
            row=2, col=1
        )
        fig.add_trace(
            go.Box(
                x=counts,
                name=str(legend_name).capitalize(),
                boxmean=True,
                showlegend=False,
                marker_color=base_color,
                opacity=0.7
                ), 
            row=1, col=1
        )

fig.update_layout(
    title="Répartition des locations par voiture (Box + Histogramme)",
    barmode="group",
    xaxis2_title="Nombre de locations pour une même voiture",
    yaxis2_title="Total",
    template="plotly_white",
    height=700
)

fig.show()

In [7]:
#répartition des delta

fig = go.Figure()

#add bar Total    
fig.add_trace(
    go.Histogram(
        x=data["time_delta_with_previous_rental_in_minutes"],
        name="Total",
        textposition='auto',
        marker_color = palette.get("Total", "gray"),
        opacity=0.7,
        nbinsx= 25
    )
)

for checkin_type, subset in data.groupby("checkin_type"):
    fig.add_trace(
        go.Histogram(
            x=subset["time_delta_with_previous_rental_in_minutes"],
            name=str(checkin_type).capitalize(),
            marker_color=palette.get(checkin_type, "gray"),
            opacity=0.7,
        )
    )

fig.update_layout(
    title="Répartition des delta temps avec les locations précédente",
    barmode="group",
    xaxis_title="Différence de temps avec la location précédente (min)",
    yaxis_title="Nombre de locations",
    template="plotly_white"
)
fig.update_xaxes(dtick=50)
fig.show()


In [8]:
analyse_data = pd.DataFrame(columns=['min_value', 'max_value', 'total', data['checkin_type'].unique()[0] , data['checkin_type'].unique()[1]])
#ajouter possibilité de changer les bornes
analyse_data['min_value']=[data['delay_at_checkout_in_minutes'].min(),-1440,-960,-480,-240,-120,-60,-30,0,30,60,120,240,480,960,1440]
analyse_data['max_value'] = analyse_data['min_value'].shift(-1)
analyse_data.loc[analyse_data.shape[0]-1, 'max_value']=data['delay_at_checkout_in_minutes'].max()
analyse_data['min_max_value'] = analyse_data.apply(lambda row: f'{row["min_value"]} - {row["max_value"]}', axis = 1)
analyse_data['total'] = analyse_data.apply(lambda row: data[(data['delay_at_checkout_in_minutes'] >= row['min_value']) & (data['delay_at_checkout_in_minutes'] < row['max_value'])]['delay_at_checkout_in_minutes'].count(), axis=1)
analyse_data[data['checkin_type'].unique()[0]] = analyse_data.apply(lambda row: data[(data['delay_at_checkout_in_minutes'] >= row['min_value']) & (data['delay_at_checkout_in_minutes'] < row['max_value']) & (data['checkin_type'] == data['checkin_type'].unique()[0])]['delay_at_checkout_in_minutes'].count(), axis=1)
analyse_data[data['checkin_type'].unique()[1]] = analyse_data.apply(lambda row: data[(data['delay_at_checkout_in_minutes'] >= row['min_value']) & (data['delay_at_checkout_in_minutes'] < row['max_value']) & (data['checkin_type'] == data['checkin_type'].unique()[1])]['delay_at_checkout_in_minutes'].count(), axis=1)


In [ ]:
display_data = analyse_data[analyse_data['min_value'] >= 0]

#répartition des delay
fig = go.Figure()

#add bar Total    
fig.add_trace(
    go.Bar(
        x=display_data["min_max_value"],
        y=display_data["total"],
        name="Total",
        text=display_data["total"],
        textposition='auto',
        marker_color = palette.get("Total", "gray"),
        opacity=0.7,
    )
)
#add bar Connect and Mobile   
for checkin_type in data['checkin_type'].unique():
    fig.add_trace(
        go.Bar(
            x=display_data["min_max_value"],
            y=display_data[checkin_type],
            name=str(checkin_type).capitalize(),
            text=display_data[checkin_type],
            textposition='auto',
            marker_color = palette.get(checkin_type, "gray"),
            opacity=0.7,
        )
    )
#Layout
fig.update_layout(
    barmode='group',
    title ='Retour en retard par rapport au checkout initial (min)',
    xaxis_title="Délai (min)",
    yaxis_title="Total"
)
fig.show()
display_data = analyse_data[analyse_data['min_value'] <= 0]

#répartition des delay
fig = go.Figure()

#add bar Total    
fig.add_trace(
    go.Bar(
        x=display_data["min_max_value"],
        y=display_data["total"],
        name="Total",
        text=display_data["total"],
        textposition='auto',
        marker_color = palette.get("Total", "gray"),
        opacity=0.7,
    )
)
#add bar Connect and Mobile   
for checkin_type in data['checkin_type'].unique():
    fig.add_trace(
        go.Bar(
            x=display_data["min_max_value"],
            y=display_data[checkin_type],
            name=str(checkin_type).capitalize(),
            text=display_data[checkin_type],
            textposition='auto',
            marker_color = palette.get(checkin_type, "gray"),
            opacity=0.7,
        )
    )
#Layout
fig.update_layout(
    barmode='group',
    title ='Retour en avance par rapport au checkout initial (min)',
    xaxis_title="Délai (min)",
    yaxis_title="Total"
)
fig.show()

In [23]:
#répartition des réservations avec une réservation antérieure

display_data = data.groupby('previous_ended_rental_id')

fig = go.Figure()

#add bar Total    
fig.add_trace(
    go.Histogram(
        x=data[data['previous_ended_rental_id'] > 0 ].groupby('car_id')["previous_ended_rental_id"].count(),
        name="Total",
        textposition='auto',
        marker_color = palette.get("Total", "gray"),
        opacity=0.7,
    )
)
#add bar Connect and Mobile   
for checkin_type, subset in data.groupby("checkin_type"):
    fig.add_trace(
        go.Histogram(
            x=subset[subset['previous_ended_rental_id'] > 0].groupby('car_id')["previous_ended_rental_id"].count(),
            name=str(checkin_type).capitalize(),
            textposition='auto',
            marker_color = palette.get(checkin_type, "gray"),
            opacity=0.7
        )
    )
#Layout
fig.update_layout(
    barmode='group',
    title ='Répartition des réservations avec une réservation antérieure par voiture',
    xaxis_title="Nbre de réservation ayant une réservation antérieure par voiture ",
    yaxis_title="Total"
)
fig.update_xaxes(dtick=1)
fig.show()

In [24]:
#répartition des locations
#histnorm='percent'
histnorm=''
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.25, 0.75],  # smaller box plot on top
    vertical_spacing=0.02
)
for checkin_type, subset in data.groupby("checkin_type"):
    for state, subset2 in subset.groupby('state'):
        counts = subset2[subset2['previous_ended_rental_id'] > 0].groupby('car_id')["previous_ended_rental_id"].size()
        legend_name = f'{checkin_type} + {state}'
        base_color = palette.get(legend_name, "gray") 
        fig.add_trace(
            go.Histogram(
                x=counts,
                name=str(legend_name).capitalize(),
                histnorm=histnorm,
                marker_color=base_color,
                showlegend=False,
                nbinsx=25,
                opacity=0.6 
            ),
            row=2, col=1
        )
        fig.add_trace(
            go.Box(
                x=counts,
                name=str(legend_name).capitalize(),
                boxmean=True,
                showlegend=False,
                marker_color=base_color,
                opacity=0.7
                ), 
            row=1, col=1
        )

fig.update_layout(
    title="Répartition des réservations avec une réservation antérieure par voiture (Box + Histogramme)",
    barmode="group",
    xaxis2_title="Nbre de réservation ayant une réservation antérieure par voiture",
    yaxis2_title="Total",
    template="plotly_white",
    height=700
)
fig.update_xaxes(dtick=1)
fig.show()

In [25]:
#répartition des locations
#histnorm='percent'
histnorm=''
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.25, 0.75],  # smaller box plot on top
    vertical_spacing=0.02
)
for checkin_type, subset in data.groupby("checkin_type"):
    for state, subset2 in subset.groupby('state'):
        counts = subset2[(subset2['delay_at_checkout_in_minutes']  > 0) & (subset2['previous_ended_rental_id'] > 0)].groupby('car_id')["previous_ended_rental_id"].size()
        legend_name = f'{checkin_type} + {state}'
        base_color = palette.get(legend_name, "gray") 
        fig.add_trace(
            go.Histogram(
                x=counts,
                name=str(legend_name).capitalize(),
                histnorm=histnorm,
                marker_color=base_color,
                showlegend=False,
                nbinsx=25,
                opacity=0.6 
            ),
            row=2, col=1
        )
        fig.add_trace(
            go.Box(
                x=counts,
                name=str(legend_name).capitalize(),
                boxmean=True,
                showlegend=False,
                marker_color=base_color,
                opacity=0.7
                ), 
            row=1, col=1
        )

fig.update_layout(
    title="Répartition des réservations avec une réservation antérieure par voiture et avec un retard (Box + Histogramme)",
    barmode="group",
    xaxis2_title="Nbre de réservation ayant une réservation antérieure par voiture et avec un retard",
    yaxis2_title="Total",
    template="plotly_white",
    height=700
)
fig.update_xaxes(dtick=1)
fig.show()

In [20]:
#répartition des locations
#histnorm='percent'
histnorm=''
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.25, 0.75],  # smaller box plot on top
    vertical_spacing=0.02
)
for checkin_type, subset in data.groupby("checkin_type"):
    for state, subset2 in subset.groupby('state'):
        display_data = subset2[(subset2['delay_at_checkout_in_minutes']  > 0) & (subset2['previous_ended_rental_id'] > 0)]["delay_at_checkout_in_minutes"]
        legend_name = f'{checkin_type} + {state}'
        base_color = palette.get(legend_name, "gray") 
        fig.add_trace(
            go.Histogram(
                x=display_data,
                name=str(legend_name).capitalize(),
                histnorm=histnorm,
                marker_color=base_color,
                showlegend=False,
                nbinsx=25,
                opacity=0.6 
            ),
            row=2, col=1
        )
        fig.add_trace(
            go.Box(
                x=display_data,
                name=str(legend_name).capitalize(),
                boxmean=True,
                showlegend=False,
                marker_color=base_color,
                opacity=0.7
                ), 
            row=1, col=1
        )

fig.update_layout(
    title="Répartition des réservations avec une réservation antérieure par voiture et avec un retard (Box + Histogramme)",
    barmode="group",
    xaxis2_title="Nbre de réservation ayant une réservation antérieure par voiture et avec un retard",
    yaxis2_title="Total",
    template="plotly_white",
    height=700
)

fig.show()

In [33]:
analyse_data = pd.DataFrame(columns=['min_value', 'max_value', 'total', data['checkin_type'].unique()[0] , data['checkin_type'].unique()[1]])
#ajouter possibilité de changer les bornes
analyse_data['min_value']=[0,30,60,120,240,480,960,1440]
analyse_data['max_value'] = analyse_data['min_value'].shift(-1)
analyse_data.loc[analyse_data.shape[0]-1, 'max_value']=data['delay_at_checkout_in_minutes'].max()
analyse_data['min_max_value'] = analyse_data.apply(lambda row: f'{row["min_value"]} - {row["max_value"]}', axis = 1)
analyse_data['total'] = analyse_data.apply(lambda row: data[(data['delay_at_checkout_in_minutes'] >= row['min_value']) & (data['delay_at_checkout_in_minutes'] < row['max_value']) & (data['previous_ended_rental_id'] > 0) ]['delay_at_checkout_in_minutes'].count(), axis=1)
analyse_data[data['checkin_type'].unique()[0]] = analyse_data.apply(lambda row: data[(data['delay_at_checkout_in_minutes'] >= row['min_value']) & (data['delay_at_checkout_in_minutes'] < row['max_value']) & (data['checkin_type'] == data['checkin_type'].unique()[0]) & (data['previous_ended_rental_id'] > 0)]['delay_at_checkout_in_minutes'].count(), axis=1)
analyse_data[data['checkin_type'].unique()[1]] = analyse_data.apply(lambda row: data[(data['delay_at_checkout_in_minutes'] >= row['min_value']) & (data['delay_at_checkout_in_minutes'] < row['max_value']) & (data['checkin_type'] == data['checkin_type'].unique()[1]) & (data['previous_ended_rental_id'] > 0)]['delay_at_checkout_in_minutes'].count(), axis=1)

#répartition des delay
fig = go.Figure()

#add bar Total    
fig.add_trace(
    go.Bar(
        x=analyse_data["min_max_value"],
        y=analyse_data["total"],
        name="Total",
        text=analyse_data["total"],
        textposition='auto',
        marker_color = palette.get("Total", "gray"),
        opacity=0.7,
    )
)
#add bar Connect and Mobile   
for checkin_type in data['checkin_type'].unique():
    fig.add_trace(
        go.Bar(
            x=analyse_data["min_max_value"],
            y=analyse_data[checkin_type],
            name=str(checkin_type).capitalize(),
            text=analyse_data[checkin_type],
            textposition='auto',
            marker_color = palette.get(checkin_type, "gray"),
            opacity=0.7,
        )
    )
#Layout
fig.update_layout(
    barmode='group',
    title ='Retour en retard par rapport au checkout initial (min) avec une location précédente',
    xaxis_title="Délai (min)",
    yaxis_title="Total"
)
fig.show()

In [40]:
analyse_data = pd.DataFrame(columns=['min_value', 'max_value', 'total', data['checkin_type'].unique()[0] , data['checkin_type'].unique()[1]])
#ajouter possibilité de changer les bornes
analyse_data['min_value']=[0,30,60,120,240,480,960,1440]
analyse_data['max_value'] = analyse_data['min_value'].shift(-1)
analyse_data.loc[analyse_data.shape[0]-1, 'max_value']=data['delay_at_checkout_in_minutes'].max()
analyse_data['min_max_value'] = analyse_data.apply(lambda row: f'{row["min_value"]} - {row["max_value"]}', axis = 1)
analyse_data['total'] = analyse_data.apply(lambda row: data[(data['delay_at_checkout_in_minutes'] >= row['min_value']) & (data['delay_at_checkout_in_minutes'] < row['max_value']) & (data['previous_ended_rental_id'].isnull()) ]['delay_at_checkout_in_minutes'].count(), axis=1)
analyse_data[data['checkin_type'].unique()[0]] = analyse_data.apply(lambda row: data[(data['delay_at_checkout_in_minutes'] >= row['min_value']) & (data['delay_at_checkout_in_minutes'] < row['max_value']) & (data['checkin_type'] == data['checkin_type'].unique()[0]) & (data['previous_ended_rental_id'].isnull())]['delay_at_checkout_in_minutes'].count(), axis=1)
analyse_data[data['checkin_type'].unique()[1]] = analyse_data.apply(lambda row: data[(data['delay_at_checkout_in_minutes'] >= row['min_value']) & (data['delay_at_checkout_in_minutes'] < row['max_value']) & (data['checkin_type'] == data['checkin_type'].unique()[1]) & (data['previous_ended_rental_id'].isnull())]['delay_at_checkout_in_minutes'].count(), axis=1)

#répartition des delay
fig = go.Figure()

#add bar Total    
fig.add_trace(
    go.Bar(
        x=analyse_data["min_max_value"],
        y=analyse_data["total"],
        name="Total",
        text=analyse_data["total"],
        textposition='auto',
        marker_color = palette.get("Total", "gray"),
        opacity=0.7,
    )
)
#add bar Connect and Mobile   
for checkin_type in data['checkin_type'].unique():
    fig.add_trace(
        go.Bar(
            x=analyse_data["min_max_value"],
            y=analyse_data[checkin_type],
            name=str(checkin_type).capitalize(),
            text=analyse_data[checkin_type],
            textposition='auto',
            marker_color = palette.get(checkin_type, "gray"),
            opacity=0.7,
        )
    )
#Layout
fig.update_layout(
    barmode='group',
    title ='Retour en retard par rapport au checkout initial (min) sans une location précédente',
    xaxis_title="Délai (min)",
    yaxis_title="Total"
)
fig.show()

In [ ]:
rental_with_previous = data[(data['previous_ended_rental_id'] > 0)]['previous_ended_rental_id'].astype(int).to_list()
matching_rows = data[data['rental_id'].isin(rental_with_previous)]

matching_rows[((matching_rows['delay_at_checkout_in_minutes'] > 0) | (matching_rows['state']== 'canceled'))].groupby('state').size()
not returning the correct value

state
ended    852
dtype: int64

In [123]:
if 565171 in data['rental_id']:
    print(True)

In [122]:
if 565171 in matching_rows['rental_id']:
    print(True)

In [62]:
data.shape[0]

21310

In [ ]:
For Predictions:
- SML with grid search 
- NN